# QC — FastQC + MultiQC

Runs QC for raw, adapter/MID `trimmed`, optional primer `pr_trimmed`, or merged FASTQ. The notebook assumes tools are installed in `/opt/conda/envs/bcr_env`.

In [ ]:
import os, sys, sysconfig
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)


In [ ]:
import subprocess
from pathlib import Path

def log(msg):
    print(f"[qc] {msg}")

def fastq_dir(volume, dataset, label):
    vol = Path(volume)
    if label == "raw":
        return vol / "raw" / dataset
    if label == "trimmed":
        return vol / "results" / dataset / "trimmed" / "fastq"
    if label == "pr_trimmed":
        return vol / "results" / dataset / "pr_trimmed" / "fastq"
    if label == "merged":
        return vol / "results" / dataset / "merged" / "fastq"
    raise ValueError("label must be raw, trimmed, pr_trimmed, or merged")

def run_qc(volume, dataset, label, threads=4):
    sd = fastq_dir(volume, dataset, label)
    if not sd.is_dir():
        raise FileNotFoundError(f"Input FASTQ directory not found: {sd}")
    base = Path(volume) / "results" / dataset / f"qc_{label}"
    fastqc_out = base / "fastqc"
    multiqc_out = base / "multiqc"
    fastqc_out.mkdir(parents=True, exist_ok=True)
    multiqc_out.mkdir(parents=True, exist_ok=True)
    fastqs = sorted(sd.glob("*.fastq.gz")) + sorted(sd.glob("*.fastq"))
    if not fastqs:
        raise FileNotFoundError(f"No FASTQ files found in {sd}")
    log(f"=== QC {dataset} ({label}): {len(fastqs)} files ===")
    subprocess.run(["fastqc", "-t", str(threads), "-q", "--noextract", *map(str, fastqs), "-o", str(fastqc_out)], check=True)
    subprocess.run(["multiqc", str(fastqc_out), "-o", str(multiqc_out), "-n", f"{dataset}_{label}_multiqc", "-f"], check=True)
    log(f"Report: {multiqc_out / (dataset + '_' + label + '_multiqc.html')}")


### Raw QC examples

These are safe to run/re-run and write `results/<dataset>/qc_raw`.

In [ ]:
for ds in ['PRJEB40348', 'PRJNA848968', 'PRJNA1247978', 'PRJNA900592']:
    run_qc('/data/user/epishkin', ds, 'raw')


### Post-trim / post-merge QC examples

Run only after generating those outputs.

In [ ]:
# Adapter/MID trim QC -> results/PRJEB40348/qc_trimmed
# run_qc('/data/user/epishkin', 'PRJEB40348', 'trimmed')
# Optional primer trim QC -> results/PRJEB40348/qc_pr_trimmed
# run_qc('/data/user/epishkin', 'PRJEB40348', 'pr_trimmed')
# Merge QC -> results/PRJEB40348/qc_merged
# run_qc('/data/user/epishkin', 'PRJEB40348', 'merged')
